In [10]:
import numpy as np
import torch
import pandas as pd
from helper import VariationalAutoencoder, load_data, train, save_params

In [11]:
known_strengths = {'null':10,'N4': 0.0, 'Q4': 1.3340727612197436, 'Q7': 2.428134794028789, 'T4': 1.9599578912997808, 'V4': 3.2307473950102388, 'G4': 4.514668716435611, 'E1': 5.21564553829087, 'A2': 0.43209185328878835, 'Y3': 0.8603530802315512}
train_strength = True

In [12]:
# Parameters for the autoencoder that can be changed

ignore_indexes=['Event','Replicate'] #ensure at least Event is ignored so that it can group properly since it is unique to each cell
group_size=100
batch_size=100
reconstruction_weight=1
strength_weight=0.0001
num_epochs=150
embedding_size=2
autoencoder_hidden_sizes=[250,200]
train_test_split=0.8
model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD38', 'CD4', 'CD44', 'CD45',
       'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
       'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
       'Proliferation', 'SSC-A', 'TBet']
# Try without CD38 (combined dataset has no CD38)
# model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD4', 'CD44', 'CD45',
#        'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
#        'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
#        'Proliferation', 'SSC-A', 'TBet']
model_path = "autoencoder_test.pt"
data_path = "../../initialSingleCellDf-channel-20220916-MW_018-001.h5"
# data_path = "../../initialSingleCellDf-channel-20220926-MW_020.h5"

def get_strength(labels,data_index_names):
       if 'Peptide' not in data_index_names:
              print("No peptide column found")
              return -1
       antigen_column = list(data_index_names).index('Peptide')
       antigen = labels[antigen_column]
       if antigen in known_strengths:
              return known_strengths[antigen]
       else:
              print("Antigen not found: ",antigen)
              return -1

In [13]:
data = pd.read_hdf(data_path, key="df")
# Add any filters here to remove data that you don't want the model to be trained on
data = data.loc[(data.index.get_level_values('CellType') == 'OT-1')]# & ((data.index.get_level_values('Time') >= 40) | (data.index.get_level_values('Peptide') == 'null'))]#& (data.index.get_level_values('Peptide') != 'T4')]

In [14]:
dataset, index_order, data_index_names, data_columns, data_labels,missing_columns = load_data(data,model_inputs,ignore_indexes,group_size,transform='normalize')
print("Missing columns: ",missing_columns)

Missing columns:  []


In [15]:
# add to the dataset the known strengths for each sample

def add_column_to_label(dataset,new_column):
    data = []
    labels = []
    for i in range(len(dataset)):
        new_labels = np.append(dataset[i][1],new_column[i])
        data.append(np.array(dataset[i][0]))
        labels.append(new_labels)
    data = np.array(data)
    labels = np.array(labels, dtype=np.float32)  # Convert labels to float32
    new_dataset = torch.utils.data.TensorDataset(torch.tensor(data),torch.tensor(labels))
    return new_dataset, len(labels[0])-1

strengths = [get_strength(labels,data_index_names) for labels in data_labels]
new_dataset, strength_index = add_column_to_label(dataset,strengths)


In [16]:
train_size = int(train_test_split * len(new_dataset))
test_size = len(new_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(new_dataset, [train_size, test_size])
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=True)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = VariationalAutoencoder(len(model_inputs),autoencoder_hidden_sizes,embedding_size).to(device)

In [18]:
train(model,train_loader,test_loader,reconstruction_weight,strength_weight,strength_index,device,num_epochs=num_epochs,train_strength=train_strength)
torch.save(model.state_dict(), model_path)
save_params(model_inputs,group_size,batch_size,embedding_size,autoencoder_hidden_sizes,model_path)

epoch [1/150], train loss:0.166819 val loss:0.118004 val reconstruction loss 0.117620 val strength loss 3.835986
epoch [2/150], train loss:0.109667 val loss:0.104508 val reconstruction loss 0.104139 val strength loss 3.690048
epoch [3/150], train loss:0.100706 val loss:0.098466 val reconstruction loss 0.098095 val strength loss 3.713692
epoch [4/150], train loss:0.095199 val loss:0.098775 val reconstruction loss 0.098393 val strength loss 3.815262
epoch [5/150], train loss:0.090777 val loss:0.089096 val reconstruction loss 0.088708 val strength loss 3.882451
epoch [6/150], train loss:0.087179 val loss:0.086861 val reconstruction loss 0.086454 val strength loss 4.063170
epoch [7/150], train loss:0.084202 val loss:0.084169 val reconstruction loss 0.083742 val strength loss 4.268960
epoch [8/150], train loss:0.081799 val loss:0.082205 val reconstruction loss 0.081785 val strength loss 4.203260
epoch [9/150], train loss:0.080338 val loss:0.080724 val reconstruction loss 0.080298 val streng